In [18]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [19]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]

In [20]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.bounds = -999999.0,999999.0
cobra_config.solver = "cplex"

In [21]:
#Loading BiGG's universal model for gapfilling

universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [22]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Cbuty.tcds.top4.gramPosN.cim8.xml"

In [23]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [24]:
#Fixing masses
model.metabolites.get_by_id("23dappa_c").formula = "C3H9N2O2"
model.metabolites.get_by_id("23dappa_e").formula = "C3H9N2O2"
model.metabolites.get_by_id("23dhb_c").formula = "C7H6O4"
model.metabolites.get_by_id("3hddecACP_c").formula = "C23H43N2O9PRS"
model.metabolites.get_by_id("3hdecACP_c").formula = "C21H39N2O9PRS"
model.metabolites.get_by_id("3hmrsACP_c").formula = "C25H47N2O9PRS"
model.metabolites.get_by_id("3hoctACP_c").formula = "C19H35N2O9PRS"
model.metabolites.get_by_id("ACP_c").formula = "C11H21N2O7PRS"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("d23hb_e").formula = "C7H7O4"
model.metabolites.get_by_id("db4p_c").formula = "C4H7O6P"
model.metabolites.get_by_id("dcaACP_c").formula = "C21H39N2O8PRS"
model.metabolites.get_by_id("ddcaACP_c").formula = "C23H43N2O8PRS"
model.metabolites.get_by_id("dmlz_c").formula = "C13H18N4O6"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fad_e").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fad_p").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmn_e").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmn_p").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("hdeACP_c").formula = "C27H49N2O8PRS"
model.metabolites.get_by_id("myrsACP_c").formula = "C25H47N2O8PRS"
model.metabolites.get_by_id("ocACP_c").formula = "C19H35N2O8PRS"
model.metabolites.get_by_id("ocdcaACP_c").formula = "C29H55N2O8PRS"
model.metabolites.get_by_id("octeACP_c").formula = "C29H53N2O8PRS"
model.metabolites.get_by_id("palmACP_c").formula = "C27H51N2O8PRS"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"
model.metabolites.get_by_id("tddec2eACP_c").formula = "C23H41N2O8PRS"
model.metabolites.get_by_id("tdec2eACP_c").formula = "C21H37N2O8PRS"
model.metabolites.get_by_id("tmrs2eACP_c").formula = "C25H45N2O8PRS"
model.metabolites.get_by_id("toct2eACP_c").formula = "C19H33N2O8PRS"

In [25]:
#Fixing charges
model.metabolites.get_by_id("23dappa_c").charge = 1
model.metabolites.get_by_id("23dappa_e").charge = 1
model.metabolites.get_by_id("23ddhb_c").charge = -1
model.metabolites.get_by_id("23dhb_c").charge = 0
model.metabolites.get_by_id("2agpg180_c").charge = -1
model.metabolites.get_by_id("2agpg180_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2maacoa_c").charge = -4
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("3hmrsACP_c").charge = -1
model.metabolites.get_by_id("3hoctACP_c").charge = -1
model.metabolites.get_by_id("3padsel_c").charge = -4
model.metabolites.get_by_id("3uib_c").charge = -1
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("adsel_c").charge = -2
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("ametam_c").charge = 2
model.metabolites.get_by_id("anhgm3p_p").charge = -2
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("db4p_c").charge = -2
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("ddcaACP_c").charge = -1
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("dtbt_c").charge = -1
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fad_e").charge = -2
model.metabolites.get_by_id("fad_p").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fc1p_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmn_e").charge = -2
model.metabolites.get_by_id("fmn_p").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("fruur_c").charge = -1
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("man6pglyc_c").charge = -3
model.metabolites.get_by_id("murein3px3p_p").charge = -4
model.metabolites.get_by_id("murein4px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4px4p_p").charge = -6
model.metabolites.get_by_id("myrsACP_c").charge = -1
model.metabolites.get_by_id("oc2coa_c").charge = -4
model.metabolites.get_by_id("octeACP_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa120_p").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa140_p").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa161_p").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa180_p").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("pa181_p").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("peptido_BS_c").charge = -2
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg160_p").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pgp120_c").charge = -3
model.metabolites.get_by_id("pgp120_p").charge = -3
model.metabolites.get_by_id("pgp140_c").charge = -3
model.metabolites.get_by_id("pgp140_p").charge = -3
model.metabolites.get_by_id("pgp160_c").charge = -3
model.metabolites.get_by_id("pgp160_p").charge = -3
model.metabolites.get_by_id("pgp161_c").charge = -3
model.metabolites.get_by_id("pgp161_p").charge = -3
model.metabolites.get_by_id("pgp180_c").charge = -3
model.metabolites.get_by_id("pgp180_p").charge = -3
model.metabolites.get_by_id("pgp181_c").charge = -3
model.metabolites.get_by_id("pgp181_p").charge = -3
model.metabolites.get_by_id("ppgpp_c").charge = -6
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("ps120_c").charge = -1
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("ps161_c").charge = -1
model.metabolites.get_by_id("ps180_c").charge = -1
model.metabolites.get_by_id("ps181_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("sbt6p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("tagur_c").charge = -1
model.metabolites.get_by_id("tddec2eACP_c").charge = -1
model.metabolites.get_by_id("tdec2eACP_c").charge = -1
model.metabolites.get_by_id("tmrs2eACP_c").charge = -1
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_e").charge = -2
model.metabolites.get_by_id("tsul_p").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2

In [26]:
# Identifying and removing duplicated reactions
# md model with removed reactions
# rd removed list
# dt reactions in doubt

[md,rd, dt] = removeDuplicateRxn(model)

In [27]:
#Removing reactions in doubt after manual inspection
md.remove_reactions([md.reactions.get_by_id("EX_abt__L_e"),md.reactions.get_by_id("EX_isetac_e"),
                     md.reactions.get_by_id("EX_glcn__D_e"),md.reactions.get_by_id("EX_ethso3_e"),
                     md.reactions.get_by_id("EX_galctr__D_e"),md.reactions.get_by_id("EX_sulfac_e"),
                     md.reactions.get_by_id("EX_orn__L_e"),md.reactions.get_by_id("EX_metsox_S__L_e"),
                     md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("DHPM1"),
                     md.reactions.get_by_id("MS_1"),md.reactions.get_by_id("SPODM_1"),
                     md.reactions.get_by_id("RBK2"),md.reactions.get_by_id("MHPGLUT")])

md.remove_metabolites([md.metabolites.get_by_id("hpglu_c"),md.metabolites.get_by_id("mhpglu_c")])

md.repair()

In [28]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
flux_variability_analysis(md,loopless=True)

,minimum,maximum
12DGR120tipp,0.000000,1.456613e-13
12DGR140tipp,0.000000,1.807776e-12
12DGR160tipp,0.000000,0.000000e+00
12DGR161tipp,0.000000,1.276756e-13
12DGR180tipp,0.000000,-1.271283e-12
...,...,...
DHORD6,0.447099,4.470994e-01
OIVD1r,0.014597,1.459718e-02
OIVD2,0.003244,3.243817e-03
SUCBZL_1,0.000135,1.351590e-04


In [29]:
#Performing gapfill with universal model from BiGG to try to reduce blocked reactions
#Usually, it does not do anything (and it takes some time to run)

gapfill(md, universal, exchange_reactions=True, demand_reactions=False, iterations=10)

[[], [], [], [], [], [], [], [], [], []]

In [30]:
#Adding/Removing/Editing reactions manually to reduce blocked reactions

md.add_reactions([universal.reactions.get_by_id("FORt2pp"),universal.reactions.get_by_id("NO2t2rpp"),
                 universal.reactions.get_by_id("ABTt")]) #Missing transport to cytoplasm

md.add_metabolites([universal.metabolites.get_by_id("agm_e")]) #Missing import from extracelular compartment
md.metabolites.get_by_id("agm_e").formula = md.metabolites.get_by_id("agm_p").formula
md.metabolites.get_by_id("agm_e").charge = md.metabolites.get_by_id("agm_p").charge
md.add_reactions([universal.reactions.get_by_id("EX_agm_e"), universal.reactions.get_by_id("AGMtex")])

md.remove_reactions([md.reactions.get_by_id("13PPDH")]) #Not connected to the network
md.remove_metabolites([md.metabolites.get_by_id("3hppnl_c"),md.metabolites.get_by_id("13ppd_c")])

md.remove_reactions([md.reactions.get_by_id("MEOHtex"),md.reactions.get_by_id("MEOHtrpp"),
                     md.reactions.get_by_id("BG_MADG"),md.reactions.get_by_id("BG_MBDG"),
                     md.reactions.get_by_id("EX_meoh_e")]) #Nothing is done with methanol in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("meoh_e"),md.metabolites.get_by_id("meoh_p"),
                       md.metabolites.get_by_id("meoh_c")])

md.remove_reactions([md.reactions.get_by_id("EX_oxa_e"),md.reactions.get_by_id("OXFOtex")]) #Nothing is done with oxalate in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("oxa_e"),md.metabolites.get_by_id("oxa_p")])

md.remove_reactions([md.reactions.get_by_id("EX_galct__D_e")]) #Nothing is done with them
md.remove_metabolites([md.metabolites.get_by_id("galct__D_e")])

md.remove_reactions([md.reactions.get_by_id("EX_met__D_e"),md.reactions.get_by_id("METte"),
                    md.reactions.get_by_id("METDabc")]) #Nothing is done with methionine in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("met__D_e"),md.metabolites.get_by_id("met__D_c")])

md.remove_reactions([md.reactions.get_by_id("EX_rmn_e"),md.reactions.get_by_id("RMNabc"),
                    md.reactions.get_by_id("RMN_Et")]) #Nothing is done with Rhamnose in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("rmn_e"),md.metabolites.get_by_id("rmn_c")])

md.reactions.get_by_id("Growth").add_metabolites({md.metabolites.get_by_id("btn_c"): -2e-06})

md.repair()

In [31]:
#Running FVA again after gapfilling
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arab__L_e,EX_arab__L_e,10,[2.625; 10],5,12.02%
arg__L_e,EX_arg__L_e,1.371,[-0.4562; 6.403],6,1.98%
asn__L_e,EX_asn__L_e,0.3258,[0; 5.977],4,0.31%
btn_e,EX_btn_e,2.703E-06,[2.568E-06; 2.658E-06],10,0.00%
ca2_e,EX_ca2_e,0.007035,[0.006683; 0.007035],0,0.00%
cl_e,EX_cl_e,0.007035,[0.006683; 0.007035],0,0.00%
cobalt2_e,EX_cobalt2_e,0.0001352,[0.0001284; 0.0001352],0,0.00%
cu2_e,EX_cu2_e,0.0009583,[0.0009104; 0.0009583],0,0.00%
cys__L_e,EX_cys__L_e,0.1246,[0; 6.796],3,0.09%
fe2_e,EX_fe2_e,0.009076,[0.008622; 0.1883],0,0.00%


In [32]:
#Loopless FBA to solve Stoichiometrically Balanced Cycles (does not help much)

rlist = [md.reactions.get_by_id("5DGLCNR"),md.reactions.get_by_id("5DKGR"),md.reactions.get_by_id("AACT"),
         md.reactions.get_by_id("ACACCT"),md.reactions.get_by_id("ACOAD1f"),md.reactions.get_by_id("ACOAD1fr"),
         md.reactions.get_by_id("ADAPAT"),md.reactions.get_by_id("ALAR"),md.reactions.get_by_id("ALATA_D"),
         md.reactions.get_by_id("ALATA_L"),md.reactions.get_by_id("ALCD19"),md.reactions.get_by_id("ALCD19y"),
         md.reactions.get_by_id("ALCD4"),md.reactions.get_by_id("ALCD4y"),md.reactions.get_by_id("ARABR"),
         md.reactions.get_by_id("ARABRr"),md.reactions.get_by_id("ASPT"),md.reactions.get_by_id("ASPTA"),
         md.reactions.get_by_id("BTS"),md.reactions.get_by_id("BTS_nadph"),md.reactions.get_by_id("CDDTPP"),
         md.reactions.get_by_id("CO2t"),md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),
         md.reactions.get_by_id("CYTDK1_1"),md.reactions.get_by_id("CYTDK2"),md.reactions.get_by_id("CYTDK2_1"),
         md.reactions.get_by_id("CYTK11_1"),md.reactions.get_by_id("DACTPP"),md.reactions.get_by_id("DAPDA"),
         md.reactions.get_by_id("DCPP"),md.reactions.get_by_id("DUCYTP"),md.reactions.get_by_id("FUM"),
         md.reactions.get_by_id("G3PD1ir"),md.reactions.get_by_id("G3PD2"),md.reactions.get_by_id("GALM1"),
         md.reactions.get_by_id("GAPD"),md.reactions.get_by_id("GAPDi_nadp"),md.reactions.get_by_id("GK1"),
         md.reactions.get_by_id("GK2"),md.reactions.get_by_id("GLBRAN2"),md.reactions.get_by_id("GLDBRAN2"),
         md.reactions.get_by_id("GLUDy"),md.reactions.get_by_id("GLUR"),md.reactions.get_by_id("GalMr"),
         md.reactions.get_by_id("GalMr_2"),md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),
         md.reactions.get_by_id("H2Otpp"),md.reactions.get_by_id("HACD1i"),md.reactions.get_by_id("HBCO_nadp"),
         md.reactions.get_by_id("HCITS"),md.reactions.get_by_id("KARA2"),md.reactions.get_by_id("KARI_1"),
         md.reactions.get_by_id("KARI_23dhmp_1"),md.reactions.get_by_id("LDH_D"),md.reactions.get_by_id("LDH_L"),
         md.reactions.get_by_id("LacR"),md.reactions.get_by_id("MDH"),md.reactions.get_by_id("MG2tex"),
         md.reactions.get_by_id("MG2tpp"),md.reactions.get_by_id("MGt5"),md.reactions.get_by_id("NADDP"),
         md.reactions.get_by_id("NADDPp_1"),md.reactions.get_by_id("NDPK1"),md.reactions.get_by_id("NDPK2"),
         md.reactions.get_by_id("NDPK4"),md.reactions.get_by_id("NDPK5"),md.reactions.get_by_id("NDPK6"),
         md.reactions.get_by_id("NDPK7"),md.reactions.get_by_id("NDPK8"),md.reactions.get_by_id("NH4t"),
         md.reactions.get_by_id("NH4tex"),md.reactions.get_by_id("NH4tpp"),md.reactions.get_by_id("OCOAT1"),
         md.reactions.get_by_id("PGCM"),md.reactions.get_by_id("PGI"),md.reactions.get_by_id("PGI1c"),
         md.reactions.get_by_id("PGMT"),md.reactions.get_by_id("SDPDS"),md.reactions.get_by_id("SDPTA"),
         md.reactions.get_by_id("SUCOAACTr"),md.reactions.get_by_id("THDPS"),md.reactions.get_by_id("THPAT"),
         md.reactions.get_by_id("UCPP"),md.reactions.get_by_id("VALTA"),md.reactions.get_by_id("VPAMTr")]         
flux_variability_analysis(md,rlist,loopless=True)

,minimum,maximum
5DGLCNR,0.000000,1000.000000
5DKGR,0.000000,1000.000000
AACT,0.000000,0.000000
ACACCT,0.000000,0.000000
ACOAD1f,0.000000,0.000000
...,...,...
THDPS,0.000000,0.013516
THPAT,0.000000,0.013516
UCPP,0.000000,0.000000
VALTA,-1.529799,0.000000


In [33]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Cbuty.tcds.top4.gramPosN.cim8.'

In [34]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Cbuty.tcds.top4.gramPosN.cim8.manual.xml
